# 07 — Primo modello multifattoriale: logistic regression OOS (Fase 4)

**Obiettivo**: il primo modello ML "vero" del progetto. Predire la **direzione**
del giorno dopo (su/giù) da una design matrix multifattoriale, valutato
**out-of-sample** con walk-forward, e confrontato onestamente con i baseline di
Fase 2.

Questo notebook **non cerca di vincere**: stabilisce se aggiungere un modello
lineare sopra le feature tecniche supera la barra del random walk / momentum.
Per CLAUDE.md: walk-forward, niente look-ahead, e documentiamo l'esito qualunque
sia.

## Ipotesi — scritte PRIMA dei risultati
1. **H1 (niente edge direzionale a breve)**: i rendimenti daily crypto sono
   ~imprevedibili (confermato in Fase 2). Mi aspetto una directional accuracy
   OOS **vicina a 0.50**, indistinguibile dal coin-flip. Un valore molto sopra
   0.50 sarebbe un campanello d'allarme di look-ahead, non un successo.
2. **H2 (lo scaler fit-on-train conta)**: standardizzare usando statistiche del
   solo train evita leakage; con poche feature lineari l'effetto sarà piccolo ma
   è metodologicamente non negoziabile.
3. **H3 (il modello lineare non batte il momentum in modo robusto)**: senza
   macro/news allineate (qui solo tecnico), l'informazione è la stessa dei
   baseline → atteso pareggio sostanziale, non un salto.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.assets.asset import get_asset_by_symbol
from src.ingestion.tier1.yahoo_finance import YahooFinanceSource
from src.features.dataset import assemble_design_matrix
from src.models.multifactor import fit_predict_walk_forward, positions_from_predictions
from src.models.baseline import (
    returns_from_prices, momentum_forecast, signal_from_forecast,
    strategy_returns, directional_accuracy,
)
from src.backtest import summarize, BINANCE_SPOT, SlippageModel, TransactionCostModel

plt.rcParams['figure.figsize'] = (11, 4)
PERIODS_PER_YEAR = 365


## 1. Dati reali + design matrix multifattoriale
Per ora **solo feature tecniche** (le macro FRED richiedono `FRED_API_KEY`, e
sono già pronte in `src.features.macro_features` per il join non appena la chiave
è disponibile). Tutte le feature sono laggate di 1 giorno (anti-look-ahead): la
riga al giorno t è lo stato alla chiusura di t-1 e predice la direzione di t.

In [2]:
src = YahooFinanceSource()
ohlcv = src.fetch_ohlcv(get_asset_by_symbol('BTC'), start='2018-01-01', interval='1d').sort_index()
X, y = assemble_design_matrix(ohlcv, feature_lag=1)
print('BTC bars:', len(ohlcv), '| design matrix:', X.shape)
print('features:', list(X.columns))
print('class balance P(up):', round(float(y.mean()), 3))
X.tail()


BTC bars: 3072 | design matrix: (3022, 5)
features: ['sma_gap', 'macd_hist', 'rsi_14', 'atr_pct', 'ret_1d']
class balance P(up): 0.51


,sma_gap,macd_hist,rsi_14,atr_pct,ret_1d
timestamp,,,,,
2026-05-26 00:00:00+00:00,0.024672,-431.885595,48.133554,0.024798,0.003882
2026-05-27 00:00:00+00:00,0.019186,-442.962567,42.711760,0.025749,-0.018817
2026-05-28 00:00:00+00:00,0.014877,-522.312977,38.015213,0.026191,-0.019532
2026-05-29 00:00:00+00:00,0.009934,-596.163908,35.707897,0.026497,-0.010870
2026-05-30 00:00:00+00:00,0.004792,-620.231653,35.240366,0.026395,-0.002231


## 2. Walk-forward OOS (expanding, train=365, test=90)
Lo scaler e il modello sono fit **solo sul train** di ogni split; le finestre di
test sono strettamente out-of-sample e coprono la timeline una volta sola.

In [3]:
res = fit_predict_walk_forward(X, y, train_size=365, test_size=90, expanding=True)
print('OOS predictions:', len(res.prediction))
print(f'Multifactor directional accuracy (OOS): {res.accuracy:.4f}')


OOS predictions: 2610
Multifactor directional accuracy (OOS): 0.4969


## 3. Confronto con i baseline di Fase 2

In [4]:
ret = returns_from_prices(ohlcv['close'])
mom = momentum_forecast(ret, lookback=30)
mom_acc = directional_accuracy(mom.reindex(ret.index), ret)
prevalence = max(float(y.mean()), 1 - float(y.mean()))
print(f'Multifactor (logistic) OOS accuracy : {res.accuracy:.4f}')
print(f'Momentum directional accuracy       : {mom_acc:.4f}')
print(f'Coin-flip baseline                  : 0.5000')
print(f'Always-majority-class baseline      : {prevalence:.4f}')


Multifactor (logistic) OOS accuracy : 0.4969
Momentum directional accuracy       : 0.5048
Coin-flip baseline                  : 0.5000
Always-majority-class baseline      : 0.5103


## 4. La strategia long-only del modello vs buy-and-hold (al netto dei costi)

In [5]:
cost = TransactionCostModel(fee=BINANCE_SPOT, slippage=SlippageModel(base_cost_bps=2.0))
pos = positions_from_predictions(res.prediction)
asset_ret = ohlcv['close'].pct_change().reindex(pos.index)
strat = strategy_returns(pos, asset_ret, cost_model=cost)
bh = asset_ret.dropna()

s_model = summarize(strat, periods_per_year=PERIODS_PER_YEAR)
s_bh = summarize(bh.reindex(strat.index).dropna(), periods_per_year=PERIODS_PER_YEAR)
comp = pd.DataFrame({
    'multifactor_net': [s_model.annualized_return, s_model.annualized_volatility,
                        s_model.sharpe, s_model.max_drawdown],
    'buy_and_hold':    [s_bh.annualized_return, s_bh.annualized_volatility,
                        s_bh.sharpe, s_bh.max_drawdown],
}, index=['ann_return', 'ann_vol', 'sharpe', 'max_drawdown'])
comp.round(3)


,multifactor_net,buy_and_hold
ann_return,0.061,0.508
ann_vol,0.494,0.621
sharpe,0.367,0.976
max_drawdown,-0.788,-0.766


## 5. Verifica ipotesi e conclusioni oneste

> Da rileggere con il modello eseguito sopra. I numeri esatti sono negli output.

- **H1 (niente edge direzionale)**: l'accuracy OOS del modello multifattoriale è
  risultata **~0.50** (coin-flip). Confermata: con sole feature tecniche, il
  daily BTC resta ~imprevedibile in direzione. È il risultato *giusto*: niente
  accuracy sospetta da look-ahead.
- **H2 (scaler fit-on-train)**: applicato per costruzione (lo `StandardScaler` è
  fit solo sul train di ogni split). Nessun leakage di preprocessing.
- **H3 (non batte il momentum)**: il modello lineare sul solo tecnico **non
  produce un edge** sopra i baseline. Atteso: l'informazione è la stessa.

**Conclusione**: il primo modello multifattoriale, con le sole feature tecniche,
**non aggiunge valore predittivo direzionale** rispetto ai baseline di Fase 2.
Questo *non* è un fallimento: è la barra onesta. Il valore di Fase 4 arriverà (se
arriverà) dall'**aggiunta di fattori ortogonali** al prezzo — macro (già pronte,
`macro_features`, point-in-time-safe) e news (Fase 3) — che portano informazione
che il tecnico da solo non ha. Il prossimo passo è il join macro+tecnico non
appena `FRED_API_KEY` è disponibile, e la valutazione se il fattore macro
sposta l'accuracy OOS oltre lo 0.50 in modo robusto.

**Bias e limiti**:
- Solo BTC, solo tecnico: survivorship + informazione limitata. Il confronto
  giusto sarà multifattoriale completo, cross-asset.
- Accuracy direzionale ≠ profittabilità: anche un piccolo edge va passato al
  netto dei costi (sezione 4) prima di crederci.
- Walk-forward expanding: i primi anni hanno meno train. Coerente con Fase 2.
